## 1️⃣ Imports & Configuration

In [1]:
# ✅ Importer les bibliothèques nécessaires
import os
import ee
import geopandas as gpd
import pandas as pd
import xarray as xr
import rioxarray
from shapely.geometry import Point
from rasterstats import zonal_stats

In [ ]:
# ✅ Définir le dossier de sortie
output_dir = r"C:\Users\tchio\OneDrive\Bureau\data science\projet biscaye\extration of climate data"
os.makedirs(output_dir, exist_ok=True)  # S'assurer que le dossier existe

print(f"✅ Tous les fichiers seront enregistrés dans : {output_dir}")

## 3️⃣ charger les fichiers NetCDF

In [6]:
# ✅ Définition des fichiers NetCDF
nc_precip = r"C:\Users\tchio\OneDrive\Bureau\data science\data on precipitation adn temp\cru_ts4.08.1901.2023.pre.dat.nc\cru_ts4.08.1901.2023.pre.dat.nc"
nc_temp = r"C:\Users\tchio\OneDrive\Bureau\data science\data on precipitation adn temp\cru_ts4.08.1901.2023.tmp.dat.nc\cru_ts4.08.1901.2023.tmp.dat.nc"


In [7]:
# ✅ Charger les NetCDF et extraire les variables
ds_precip = xr.open_dataset(nc_precip)
ds_temp = xr.open_dataset(nc_temp)

In [8]:
# ✅ Sélection des variables
da_precip = ds_precip["pre"]
da_temp = ds_temp["tmp"]

In [ ]:
print(ds_precip)  # Afficher la structure du fichier de précipitations
print(ds_temp)    # Afficher la structure du fichier de températures


## 4️⃣ Chargement de la grille et des mines

In [ ]:
# ✅ Définir le chemin de la grille subdivisée (mise à jour)
grid_file = r"C:\Users\tchio\OneDrive\Bureau\data science\projet biscaye\extration of climate data\reduce_10km_grid\reduced_10km_grid.shp"

# ✅ Charger la grille subdivisée avec Geopandas
grid_gdf = gpd.read_file(grid_file)

# ✅ Vérifier le chargement
print(f"✅ Grille subdivisée chargée avec succès ! Nombre de cellules : {len(grid_gdf)}")
grid_gdf.head()  # Afficher les premières lignes


In [11]:
# ✅ Charger les données des mines
file_path = r"C:\Users\tchio\OneDrive\Bureau\data science\projet biscaye\extraire coordonnées géo_mines\données_geocodés\données_geocodés\Données géocodées\données_geocodés\merged_data.xlsx"
mine_locations = pd.read_excel(file_path, sheet_name="Sheet1")

In [12]:
# ✅ Supprimer les doublons
mine_locations = mine_locations.drop_duplicates(subset=["Mine", "Latitude", "Longitude"]).dropna().reset_index(drop=True)


In [ ]:
# ✅ Convertir en GeoDataFrame
mine_gdf = gpd.GeoDataFrame(
    mine_locations,
    geometry=[Point(lon, lat) for lat, lon in zip(mine_locations["Latitude"], mine_locations["Longitude"])],
    crs="EPSG:4326"
)

print(f"✅ Mines chargées : {len(mine_gdf)} sites miniers trouvés.")

##  5️⃣ Association des mines aux cellules

In [14]:
# ✅ Associer chaque mine à la cellule où elle se trouve
mines_with_cells = gpd.sjoin(mine_gdf, grid_gdf, how="left", predicate="within")

In [15]:
# ✅ Vérifier si certaines mines ne sont pas associées à une cellule
missing_mines = mines_with_cells[mines_with_cells["index_right"].isna()]
if not missing_mines.empty:
    print(f"⚠ {len(missing_mines)} mines ne sont pas dans la grille !")

In [ ]:
# ✅ Supprimer les mines non associées
mines_with_cells = mines_with_cells.dropna(subset=["index_right"])

print(f"✅ Mines associées aux cellules : {len(mines_with_cells)} mines traitées.")

## 6️⃣ Extraction des données climatiques pour chaque mine

In [ ]:
# ✅ Définir les années d'extraction
start_year = 1901  # Modifier si nécessaire
end_year = 2023

# ✅ Vérification des années
print(f"✅ Extraction des données climatiques mensuelles de {start_year} à {end_year}...")

# ✅ Initialiser une liste pour stocker les données mensuelles
climate_records = []

print(f"✅ Début de l'extraction des données climatiques mensuelles de {start_year} à {end_year}...")

# ✅ Vérifier que la plage temporelle est correcte
print(f"📅 Première date du NetCDF : {str(da_precip['time'].values[0])[:7]}")
print(f"📅 Dernière date du NetCDF : {str(da_precip['time'].values[-1])[:7]}")
print(f"✅ Nombre total de mois disponibles : {len(da_precip['time'])}")

# ✅ Parcourir chaque mois dans la plage temporelle
for i, time_label in enumerate(da_precip["time"].values):
    date_label = str(time_label)[:7]  # Format YYYY-MM
    print(f"🔄 Extraction des données pour {date_label}...")
    
    # ✅ Vérification : S'assurer que la date est bien dans la plage attendue
    if date_label < "1901-01" or date_label > "2023-12":
        print(f"❌ Erreur : La date {date_label} est hors de la plage attendue !")
        continue  # Passer à la prochaine itération si la date est incorrecte

    # ✅ Extraire la couche de précipitations et de températures pour le mois courant
    precip_month = da_precip.isel(time=i)
    temp_month = da_temp.isel(time=i)
    
    # ✅ Définir les chemins des fichiers temporaires GeoTIFF
    precip_tif = os.path.join(output_dir, f"temp_precip_{date_label}.tif")
    temp_tif = os.path.join(output_dir, f"temp_temp_{date_label}.tif")
    
    # ✅ Vérifier et définir les dimensions spatiales avant conversion en raster
    precip_month = precip_month.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    temp_month = temp_month.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    
    # ✅ Assurer que le CRS est défini correctement
    precip_month = precip_month.rio.write_crs("EPSG:4326", inplace=False)
    temp_month = temp_month.rio.write_crs("EPSG:4326", inplace=False)
    
    # ✅ Convertir en GeoTIFF temporaire
    precip_month.rio.to_raster(precip_tif)
    temp_month.rio.to_raster(temp_tif)
    
    # ✅ Appliquer l'extraction des valeurs climatiques pour chaque mine
    precip_stats = zonal_stats(mines_with_cells, precip_tif, stats=["mean"], nodata=-9999)
    temp_stats = zonal_stats(mines_with_cells, temp_tif, stats=["mean"], nodata=-9999)
    
    # ✅ Ajouter les données extraites à la liste
    for j, row in mines_with_cells.iterrows():
        climate_records.append({
            "Mine": row["Mine"],
            "Latitude": row.geometry.y,
            "Longitude": row.geometry.x,
            "Date": date_label,  # Format YYYY-MM
            "Precipitation": precip_stats[j]["mean"] if precip_stats[j]["mean"] is not None else 0,
            "Temperature": temp_stats[j]["mean"] if temp_stats[j]["mean"] is not None else 0
        })
    
    # ✅ Supprimer les fichiers temporaires après utilisation
    os.remove(precip_tif)
    os.remove(temp_tif)

print("✅ Extraction mensuelle des précipitations et températures terminée.")

# ✅ Convertir les données en DataFrame
climate_df = pd.DataFrame(climate_records)

# ✅ Trier les données pour un bon ordre temporel
climate_df = climate_df.sort_values(by=["Mine", "Date"]).reset_index(drop=True)

# ✅ Sauvegarder les données en CSV
climate_output_file = os.path.join(output_dir, "mine_climate_data_monthly_1901_2023.csv")
climate_df.to_csv(climate_output_file, index=False)
print(f"✅ Extraction terminée ! Données enregistrées sous {climate_output_file}")
